# Make Datasets
After the fuzzy matching, we create some datasets. We focus on the *both* group, where song title and artist match.

In [ ]:
import os
from tqdm import tqdm
import pandas as pd

subdir = "qwen_all" # can be "", "llama" or "qwen"
matched_path = os.path.join(os.path.join("data", "matched", subdir))

# collect selected (only where artist AND title match)
filtered_types = pd.read_csv(os.path.join(matched_path, "filtered_types.csv"), sep="\t")
selected = filtered_types.reset_index()
selected = selected.loc[selected["match_type"] == "both", ["clique_id", "youtube_id"]]
selected = selected.sort_values(by=["clique_id", "youtube_id"])


# Exploring some ideas to filter
Goal: get rather non-official versions such as amateur covers.

Ideas:
- filter by stopwords (e.g. "official")
- filter by channels (e.g. )
- filter by ambiguity of entities (e.g. the artist "the the" did not work well when fuzzy matching)

In [10]:
metadata = pd.read_json(os.path.join("data", "metadata_filtered.jsonl"), lines=True, orient='records')
metadata = metadata.loc[metadata.id.isin(selected.youtube_id),:]

new_columns = ["clique_id"] + metadata.columns.tolist()
metadata = pd.merge(
    selected.reset_index(),
    metadata,
    left_on="youtube_id",
    right_on="id",
    how="left",
)
metadata = metadata[new_columns]
metadata.loc[:,"channel_name"] = metadata.channel.apply(lambda x: x["name"])


## Filter by stopwords

In [37]:
import re

stopwords = ["remaster", "official", "music video", "lyric video"]

pattern = "|".join(re.escape(word) for word in stopwords)

mask_stopwords = metadata["title"].str.contains(pattern, case=False, na=False)
print(f"Removed {mask_stopwords.sum()} entries with stopwords in title")


Removed 49279 entries with stopwords in title


## By channel name
- remove where endswith `VEVO`
- remove where `lyric videos` is contained
- remove `- Topic` channels. See: https://support.google.com/youtube/answer/7636475?hl=en#zippy=%2Chow-does-youtube-decide-when-to-auto-generate-a-topic-channel-for-an-artist



In [38]:
# filtering masks
mask_vevo = metadata.channel_name.str.endswith("VEVO")
mask_lyric_videos = metadata.channel_name.str.contains("lyric video", case=False, na=False)
mask_topic = metadata.channel_name.str.endswith("- Topic")

print(f"Removed {mask_vevo.sum()} VEVO entries, {mask_lyric_videos.sum()} lyric video entries, and {mask_topic.sum()} Topic entries")


Removed 2902 VEVO entries, 266 lyric video entries, and 382092 Topic entries


In [39]:
selected_filtered = metadata.loc[
    ~mask_stopwords & ~mask_vevo & ~mask_lyric_videos & ~mask_topic,
    ["clique_id", "id"],
].rename({"id": "youtube_id"}, axis=1)

print(f"Resulting dataset has {len(selected_filtered)} entries.")


Resulting dataset has 883515 entries.


## Make datasets
We write files with new IDs only and joint with the first version of `Discogs-VI-YT`.


In [40]:
selected_full = selected.groupby("clique_id").agg(list)
selected_filtered = selected_filtered.groupby("clique_id").agg(list)

dataset_path = os.path.join("data", "dataset", subdir)
# write to json
selected_full.to_json(
    os.path.join(dataset_path, "selected_versions.json"), orient="index")
selected_filtered.to_json(
    os.path.join(dataset_path, "selected_versions_filtered.json"), orient="index")


In [41]:
# transform to dict
selected_full = selected_full.to_dict(orient="index")
selected_filtered = selected_filtered.to_dict(orient="index")


In [ ]:
import json
import numpy as np
from tqdm import tqdm

def read_jsonl(path: str) -> list:
    """
    Read the first k lines from a JSONL file.
    """
    results = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            results.append(json.loads(line))
    return results

def update_discogs_with_youtube_ids(
    input_path: str,
    output_path: str,
    selected: dict
) -> dict:
    """
    Updates a Discogs JSONL file with new YouTube version entries from `selected`.

    Args:
        input_path (str): Path to the original Discogs JSONL file.
        output_path (str): Path to save the updated Discogs JSONL file.
        selected (dict): Dictionary mapping `clique_id` to YouTube IDs:
                         e.g., {'C-0000023': {'youtube_id': ['abc123', ...]}}

    Returns:
        dict: Summary statistics including affected cliques and version counts.
    """

    discogs_updated = []
    versions_before = []
    versions_after = []
    new_versions_total = 0
    affected_cliques = 0

    data_in = read_jsonl(input_path)
    print(f"Input file {input_path} has {len(data_in)} lines.")
    
    with open(output_path, "w") as fout:
        print("Processing..")
        counter = 0
        for clique in data_in:
            print(f"\rProcessing line {counter:<10}", end="", flush=True)
            clique_id = clique["clique_id"]

            n_before = len(clique.get("versions", []))

            if clique_id in selected:
                new_youtube_ids = selected[clique_id]["youtube_id"]
                for youtube_id in new_youtube_ids:
                    clique["versions"].append(
                        {
                            "version_id": youtube_id,
                            "youtube_video": [
                                {
                                    "url": f"https://www.youtube.com/watch?v={youtube_id}",
                                    "source": "youtube_query",
                                    "match_type": 100
                                }
                            ]
                        }
                    )
                new_versions_total += len(new_youtube_ids)
                print(f"\rAdded {len(new_youtube_ids)} versions", end="", flush=True)

                affected_cliques += 1

            n_after = len(clique["versions"])
            versions_before.append(n_before)
            versions_after.append(n_after)
            
            discogs_updated.append(clique)
            fout.write(json.dumps(clique) + "\n")
            counter += 1
            
    def describe(arr):
        return {
            "min": int(np.min(arr)),
            "max": int(np.max(arr)),
            "mean": float(np.mean(arr)),
            "median": float(np.median(arr)),
        }

    print("=== Version Statistics Summary ===")
    print(f"Total cliques processed: {len(versions_before)}")
    print(f"Total cliques affected (added YT IDs): {affected_cliques}")
    print(f"Total new YouTube versions added: {new_versions_total}")
    print("\nVersions per clique (before):", describe(versions_before))
    print("Versions per clique (after):", describe(versions_after))

discogs_path = "data/discogs/dvi_cleaned.jsonl"

discogs2_path = os.path.join(dataset_path, f"dvi_fm.jsonl")  # FM = Fuzzy Match
discogs2_path_filtered = os.path.join(dataset_path, f"dvi_fm_filtered.jsonl")  # FM = Fuzzy Match

update_discogs_with_youtube_ids(discogs_path, discogs2_path_filtered, selected_filtered)
update_discogs_with_youtube_ids(discogs_path, discogs2_path, selected_full)


Input file data/discogs/Discogs-VI-YT-20240701_qwen_all.jsonl.cleaned has 128984 lines.
Processing..
Processing line 128983    === Version Statistics Summary ===
Total cliques processed: 128984
Total cliques affected (added YT IDs): 56497
Total new YouTube versions added: 883515

Versions per clique (before): {'min': 1, 'max': 629, 'mean': 3.822559387210817, 'median': 2.0}
Versions per clique (after): {'min': 1, 'max': 629, 'mean': 10.672362463561372, 'median': 3.0}
Input file data/discogs/Discogs-VI-YT-20240701_qwen_all.jsonl.cleaned has 128984 lines.
Processing..
Processing line 128983    === Version Statistics Summary ===
Total cliques processed: 128984
Total cliques affected (added YT IDs): 64532
Total new YouTube versions added: 1298169

Versions per clique (before): {'min': 1, 'max': 629, 'mean': 3.822559387210817, 'median': 2.0}
Versions per clique (after): {'min': 1, 'max': 629, 'mean': 13.887133287849656, 'median': 4.0}
